In [94]:
import json
import time

import numpy as np
import pandas as pd

from bs4 import BeautifulSoup

from pydantic import BaseModel
from typing import List, Optional

from selenium import webdriver

from supabase import create_client, Client

In [95]:
driver = webdriver.Chrome()

In [ ]:
whoscored_url = "https://www.whoscored.com/matches/1887332/live/international-africa-cup-of-nations-2025-morocco-comoros"
out_name = "4-morocco_tanzania_events.csv"
from selenium import webdriver
from selenium.webdriver.chrome.options import Options

chrome_options = Options()
# This is the "magic" line that connects to your manual browser
chrome_options.add_experimental_option("debuggerAddress", "127.0.0.1:9222")

driver = webdriver.Chrome(options=chrome_options)

# If you already have WhoScored open in that window, you don't even need .get()
# But calling it now should work because you have your real cookies/fingerprint
driver.get(whoscored_url)

In [87]:
soup = BeautifulSoup(driver.page_source, 'html.parser')

In [88]:
element = soup.select_one('script:-soup-contains("matchCentreData")')

In [92]:
matchdict = json.loads(element.text.split("matchCentreData: ")[1].split(',\n')[0])

In [93]:
matchdict.keys()


dict_keys(['playerIdNameDictionary', 'periodMinuteLimits', 'timeStamp', 'attendance', 'venueName', 'referee', 'weatherCode', 'elapsed', 'startTime', 'startDate', 'score', 'htScore', 'ftScore', 'etScore', 'pkScore', 'statusCode', 'periodCode', 'home', 'away', 'maxMinute', 'minuteExpanded', 'maxPeriod', 'expandedMinutes', 'expandedMaxMinute', 'periodEndMinutes', 'commonEvents', 'events', 'timeoutInSeconds'])

In [66]:
match_events = matchdict['events']

In [67]:
df = pd.DataFrame(match_events)

In [68]:
df.columns

Index(['id', 'eventId', 'minute', 'teamId', 'playerId', 'x', 'y',
       'expandedMinute', 'period', 'type', 'outcomeType', 'qualifiers',
       'satisfiedEventsTypes', 'isTouch', 'second', 'endX', 'endY',
       'relatedEventId', 'relatedPlayerId', 'blockedX', 'blockedY',
       'goalMouthZ', 'goalMouthY', 'isShot', 'cardType', 'isGoal'],
      dtype='str')

In [69]:
df.dropna(subset='playerId', inplace=True)

In [70]:
df = df.where(pd.notnull(df), None)

In [71]:
df = df.rename(
    {
        'eventId': 'event_id',
        'expandedMinute': 'expanded_minute',
        'outcomeType': 'outcome_type',
        'isTouch': 'is_touch',
        'playerId': 'player_id',
        'teamId': 'team_id',
        'endX': 'end_x',
        'endY': 'end_y',
        'blockedX': 'blocked_x',
        'blockedY': 'blocked_y',
        'goalMouthZ': 'goal_mouth_z',
        'goalMouthY': 'goal_mouth_y',
        'isShot': 'is_shot',
        'cardType': 'card_type',
        'isGoal': 'is_goal'
    },
    axis=1
)

In [72]:
df['period_display_name'] = df['period'].apply(lambda x: x['displayName'])
df['type_display_name'] = df['type'].apply(lambda x: x['displayName'])
df['outcome_type_display_name'] = df['outcome_type'].apply(lambda x: x['displayName'])

In [73]:
df.drop(columns=["period", "type", "outcome_type"], inplace=True)

In [74]:
if 'is_goal' not in df.columns:
    print('missing goals')
    df['is_goal'] = False

In [75]:
df = df[~(df['type_display_name'] == "OffsideGiven")]

In [76]:
df.columns

Index(['id', 'event_id', 'minute', 'team_id', 'player_id', 'x', 'y',
       'expanded_minute', 'qualifiers', 'satisfiedEventsTypes', 'is_touch',
       'second', 'end_x', 'end_y', 'relatedEventId', 'relatedPlayerId',
       'blocked_x', 'blocked_y', 'goal_mouth_z', 'goal_mouth_y', 'is_shot',
       'card_type', 'is_goal', 'period_display_name', 'type_display_name',
       'outcome_type_display_name'],
      dtype='str')

In [77]:
df.head()

,id,event_id,minute,team_id,player_id,x,y,expanded_minute,qualifiers,satisfiedEventsTypes,...,blocked_x,blocked_y,goal_mouth_z,goal_mouth_y,is_shot,card_type,is_goal,period_display_name,type_display_name,outcome_type_display_name
3,2.885533e+09,3,0,495,317541.0,49.9,50.1,0,"[{'type': {'value': 141, 'displayName': 'PassE...","[91, 117, 30, 35, 37, 215, 218]",...,NaN,NaN,NaN,NaN,None,None,None,FirstHalf,Pass,Successful
4,2.885533e+09,4,0,495,236519.0,30.5,63.1,0,"[{'type': {'value': 140, 'displayName': 'PassE...","[91, 120, 128, 36, 38, 216, 218]",...,NaN,NaN,NaN,NaN,None,None,None,FirstHalf,Pass,Unsuccessful
5,2.885533e+09,3,0,9719,398417.0,33.6,94.3,0,"[{'type': {'value': 233, 'displayName': 'Oppos...","[197, 200]",...,NaN,NaN,NaN,NaN,None,None,None,FirstHalf,Aerial,Successful
6,2.885533e+09,5,0,495,349932.0,66.4,5.7,0,"[{'type': {'value': 286, 'displayName': 'Offen...","[198, 199]",...,NaN,NaN,NaN,NaN,None,None,None,FirstHalf,Aerial,Unsuccessful
7,2.885533e+09,4,0,9719,398417.0,33.6,94.3,0,"[{'type': {'value': 56, 'displayName': 'Zone'}...","[91, 94, 95, 57, 216]",...,NaN,NaN,NaN,NaN,None,None,None,FirstHalf,Clearance,Successful


In [78]:
df.qualifiers.iloc[0]

[{'type': {'value': 141, 'displayName': 'PassEndY'}, 'value': '64.5'},
 {'type': {'value': 213, 'displayName': 'Angle'}, 'value': '2.72'},
 {'type': {'value': 140, 'displayName': 'PassEndX'}, 'value': '29.0'},
 {'type': {'value': 56, 'displayName': 'Zone'}, 'value': 'Back'},
 {'type': {'value': 212, 'displayName': 'Length'}, 'value': '24.0'}]

In [79]:
flat_qualifiers = []

for row in df['qualifiers']:
    row_data = {}
    if isinstance(row, list):
        for item in row:
            # displayName becomes the column name
            name = item.get('type', {}).get('displayName')
            # 'value' is the data; if it's missing, it's a 'tag' (so we mark as True)
            val = item.get('value', True)
            if name:
                row_data[name] = val
    flat_qualifiers.append(row_data)

# 2. Create the new DataFrame (Pandas handles the NaNs automatically)
qual_df = pd.DataFrame(flat_qualifiers, index=df.index)

# 3. Concatenate and convert numeric strings to actual floats/ints
df = pd.concat([df, qual_df], axis=1)

for col in qual_df.columns:
    try:
        # If it's a number, convert it.
        df[col] = pd.to_numeric(df[col])
    except (ValueError, TypeError):
        # If it's a string like 'Back' or 'Zone', leave it as is.
        continue

In [80]:
df.columns

Index(['id', 'event_id', 'minute', 'team_id', 'player_id', 'x', 'y',
       'expanded_minute', 'qualifiers', 'satisfiedEventsTypes',
       ...
       'BigChanceCreated', 'HighClaim', 'IntentionalGoalAssist', 'OverRun',
       'Volley', 'HighCentre', 'BoxRight', 'Collected', 'LeadingToGoal',
       'LastMan'],
      dtype='str', length=105)

In [81]:
df.shape[1]

105

In [82]:
df.columns

Index(['id', 'event_id', 'minute', 'team_id', 'player_id', 'x', 'y',
       'expanded_minute', 'qualifiers', 'satisfiedEventsTypes',
       ...
       'BigChanceCreated', 'HighClaim', 'IntentionalGoalAssist', 'OverRun',
       'Volley', 'HighCentre', 'BoxRight', 'Collected', 'LeadingToGoal',
       'LastMan'],
      dtype='str', length=105)

In [83]:
list(df)

['id',
 'event_id',
 'minute',
 'team_id',
 'player_id',
 'x',
 'y',
 'expanded_minute',
 'qualifiers',
 'satisfiedEventsTypes',
 'is_touch',
 'second',
 'end_x',
 'end_y',
 'relatedEventId',
 'relatedPlayerId',
 'blocked_x',
 'blocked_y',
 'goal_mouth_z',
 'goal_mouth_y',
 'is_shot',
 'card_type',
 'is_goal',
 'period_display_name',
 'type_display_name',
 'outcome_type_display_name',
 'PassEndY',
 'Angle',
 'PassEndX',
 'Zone',
 'Length',
 'Longball',
 'OppositeRelatedEvent',
 'Defensive',
 'Offensive',
 'Head',
 'ThrowIn',
 'StandingSave',
 'PlayerCaughtOffside',
 'IndirectFreekickTaken',
 'FreekickTaken',
 'Chipped',
 'HeadPass',
 'LayOff',
 'MissHigh',
 'GoalKick',
 'Cross',
 'RightFoot',
 'IntentionalAssist',
 'ShotAssist',
 'KeyPass',
 'BlockedX',
 'Assisted',
 'Blocked',
 'GoalMouthZ',
 'RegularPlay',
 'GoalMouthY',
 'FirstTouch',
 'BlockedY',
 'RelatedEventId',
 'BoxCentre',
 'LowCentre',
 'OutfielderBlock',
 'Foul',
 'Penalty',
 'KeeperSaved',
 'MissRight',
 'DivingSave',
 'Lo

In [84]:
def output_to_csv(daf, filename):
    if df is not None:
        df.to_csv(filename, index=False)
        print(f"--- SUCCESS --- Saved to {filename}")


In [85]:
output_to_csv(daf=df, filename=out_name)

--- SUCCESS --- Saved to 3-morocco_zambia_events.csv
